In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

In [4]:
token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.github.ai/inference"
model = "openai/gpt-4.1-mini"

In [5]:
llm = ChatOpenAI(
    model=model,
    api_key=token,
    base_url=endpoint
)

In [6]:
from langgraph.types import Command

In [7]:
from langgraph.prebuilt import create_react_agent

In [8]:
def add_number(state):
    result = state["num1"]+state["num2"]
    print(f"addition is {result}")
    return Command(goto="multiply", update={"sum":result})

In [9]:
state = {"num1":10, "num2":20}

In [10]:
add_number(state)

addition is 30


Command(update={'sum': 30}, goto='multiply')

Creating one dummy multiagent

In [11]:
from typing import Annotated,Sequence, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

In [12]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    name:str
    age: int
    DOB: int

In [13]:
def transfer_to_multiplication_expert():
    "Ask multiplicatin agent for a help"
    return

In [14]:
def transfer_to_additon_expert():
    "Ask addition agent for a help"
    return

In [15]:
llm_with_tools = llm.bind_tools([transfer_to_multiplication_expert])

In [16]:
response = llm_with_tools.invoke("hi")

In [17]:
response.content

'Hello! How can I assist you today?'

In [18]:
response.tool_calls

[]

In [19]:
response = llm_with_tools.invoke("what is 2 multiply by 2")

In [20]:
response.tool_calls

[{'name': 'transfer_to_multiplication_expert',
  'args': {},
  'id': 'call_Iuyet4WA5vHMQaqOSLXwzDJO',
  'type': 'tool_call'}]

In [21]:
from typing_extensions import Literal
from langgraph.graph import MessagesState, StateGraph, START, END

In [22]:
system_prompt = (
        "you are an addition expert , you can ask the multiplication expert for help with multiplicaiton"
        "always to you portion of a calculation before handoff"
    )
  

In [23]:
messages = [{"role":"system", "content":system_prompt}]+ ["Can you tell me addtion of 2 and 2"]

In [24]:
messages

[{'role': 'system',
  'content': 'you are an addition expert , you can ask the multiplication expert for help with multiplicaitonalways to you portion of a calculation before handoff'},
 'Can you tell me addtion of 2 and 2']

In [25]:
def additional_expert(state:MessagesState)-> Command[Literal["additional_expert","__end__"]]:
    system_prompt = (
        "you are an addition expert , you can ask the multiplication expert for help with multiplicaiton"
        "always to you portion of a calculation before handoff"
    )
    messages = [{"role":"system", "content":system_prompt}]+ state["messages"]

    llm_with_tools= llm.bind_tools({transfer_to_multiplication_expert})
    ai_message= llm_with_tools.invoke(messages)

    if len(ai_message.tool_calls)>0:
        tool_call_id = ai_message.tool_calls[-1]["id"]
        tool_message={
            "role":"tool",
            "content":"Successfully transferred",
            "tool_call_id":tool_call_id,
        }

        return Command(
            goto="multiplication_expert",update={"messages":[ai_message,tool_message]}
        )
    return {"messages":[ai_message]}

In [26]:
def multiplication_expert(state:MessagesState)-> Command[Literal["multiplication_expert","__end__"]]:
    system_prompt = (
        "you are an multiplication expert , you can ask the multiplication expert for help with multiplicaiton"
        "always to you portion of a calculation before handoff"
    )
    messages = [{"role":"system", "content":system_prompt}]+ state["messages"]

    llm_with_tools= llm.bind_tools({transfer_to_multiplication_expert})
    ai_message= llm_with_tools.invoke(messages)

    if len(ai_message.tool_calls)>0:
        tool_call_id = ai_message.tool_calls[-1]["id"]
        tool_message={
            "role":"tool",
            "content":"Successfully transferred",
            "tool_call_id":tool_call_id,
        }

        return Command(
            goto="additional_expert",update={"messages":[ai_message,tool_message]}
        )
    return {"messages":[ai_message]}

In [27]:
graph= StateGraph(MessagesState)

In [28]:
graph.add_node("additional_expert", additional_expert)

In [29]:
graph.add_node("multiplication_expert", multiplication_expert)

In [30]:
graph.add_edge(START,"additional_expert")

In [31]:
app= graph.compile()

In [32]:
app.invoke({"message":[("user","whats (3+5)*12. Provide me the output")]})

{'messages': [AIMessage(content="Hello! I can help you with addition problems. If your problem involves multiplication, I can ask the multiplication expert for assistance before continuing with the addition part. Please provide the addition problem you'd like help with!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 69, 'total_tokens': 112, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c82f8028fa', 'id': 'chatcmpl-CNFOYUjJE6K6PB0ZmKQuMFU21H8VU', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--26856a45-1844-4470-bd02-ddf04a3a2608-0', usage_metadata={'input_tokens': 69, 'output_tokens': 43, 'total_tokens': 112, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu

Wtih realtime tool

In [33]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name in english?")

'Barack Obama was born on August 4, 1961,[2] at Kapiolani Medical Center for Women and Children in Honolulu, Hawaii.[3][4][5][6] He is the only president born outside the contiguous 48 states.[7] He was born to an 18-year-old American mother and a 27-year-old Kenyan father. англ. Barack Hussein Obama II[ 1 ]. Отец. Барак Хуссейн Обама — старший. Obama \' s First Retrospective Job Approval Rating Is 63%. Институт Гэллапа (англ.). Barack Obama ’ s parents married while students at the University of Hawaii. His father, Barack Obama , Sr., a Kenyan, became an economist in the government of Kenya. His mother, S . Ann Dunham, became an anthropologist. They divorced in 1964. Ann then married (and later divorced) another foreign student, Indonesian Lolo Soetoro. Where did Barack Obama attend school? Barack Obama graduated from Punahou School, an elite academy in Honolulu, and then attended Occidental College before transferring to Columbia University and earning (1983) a B.A. in political scie

In [35]:
from langchain_experimental.utilities import PythonREPL

In [36]:
repl= PythonREPL()

In [37]:
code = """
x=3,
y=5,
print(x+y)
"""

In [38]:
repl.run(code)

Python REPL can execute arbitrary code. Use with caution.


'(3, 5)\n'

In [ ]:
@tool
def python_repl_tool():
    """
    
    """
    repl = PythonREPL()
    repl.run()


In [ ]:
def research_node(state:MessagesState)->Command[Literal["chart_generator",END]]:
    create_react_agent(

        llm, 
        tools = [search_tool], 
        prompt= make_system_prompt(
            
        )

    )

In [ ]:
def chart_node(state:MessagesState)-> Command[Literal["researcher", END]]:
    pass